# Type Ia Supernova MonitorMonitors ANTARES and ALeRCE brokers for Type Ia supernova candidates.  ANTARES aggregates **ZTF + Rubin/LSST** data (ATLAS not available).  Edit the parameters below and **Restart Kernel & Run All**.

In [ ]:
# ========================= PARAMETERS =========================MIN_IA_PROBABILITY = 0.5   # Min Type Ia probability (0-1)DAYS_BACK          = 30    # How many days back to searchMAX_ALERTS         = 200   # Max alerts per brokerREQUIRE_RUBIN      = True  # Only keep loci with Rubin/LSST dataFILTER_ELLIPTICAL  = False # Filter for elliptical host galaxiesMIN_BROKERS        = 1     # Min brokers detecting object (1 or 2)MIN_AGREEMENT      = 0.0   # Min agreement score (0.0 = disabled)CACHE_DIR          = './cache/data'# ==============================================================

In [ ]:
import sys, ossys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('supernova_monitor.ipynb'))))import warnings; warnings.filterwarnings('ignore')import logging, pandas as pd, numpy as npfrom datetime import datetimefrom IPython.display import display, HTML%matplotlib inlineimport matplotlib; matplotlib.rcParams['figure.dpi'] = 100import matplotlib.pyplot as pltlogging.basicConfig(level=logging.INFO)from supernova_monitor import SupernovaMonitorfrom utils.plotting import PlottingUtilsmonitor = SupernovaMonitor(cache_dir=CACHE_DIR)print(f"Active brokers: {', '.join(monitor.brokers.keys())}")

In [ ]:
results = monitor.run_full_pipeline(    min_ia_probability=MIN_IA_PROBABILITY,    days_back=DAYS_BACK,    limit=MAX_ALERTS,    require_rubin=REQUIRE_RUBIN,    filter_elliptical=FILTER_ELLIPTICAL,    min_agreement=MIN_AGREEMENT,    min_brokers=MIN_BROKERS,)if results is not None and len(results) > 0:    candidates = results    print(f"Found {len(candidates)} candidates")    if 'mean_ia_prob' in candidates.columns:        v = candidates['mean_ia_prob'].dropna()        if len(v) > 0:            print(f"  P(Ia): mean={v.mean():.3f}  min={v.min():.3f}  max={v.max():.3f}")    if 'num_brokers' in candidates.columns:        for n, c in candidates['num_brokers'].value_counts().sort_index().items():            print(f"  {n} broker(s): {c} objects")else:    candidates = None    print("No candidates found.")    if REQUIRE_RUBIN:        print("Try REQUIRE_RUBIN=False to include ZTF-only objects.")

In [ ]:
if candidates is not None and len(candidates) > 0:
    display_cols = [c for c in [
        'rubin_dia_object_id', 'object_id_ANTARES', 'object_id_ALeRCE',
        'ra', 'dec', 'discovery_date', 'brokers_detected', 'mean_ia_prob',
        'agreement_score', 'magnitude_ANTARES', 'magnitude_ALeRCE',
    ] if c in candidates.columns]
    fmt = {}
    for c in display_cols:
        if c in ('ra','dec'): fmt[c] = '{:.5f}'
        elif 'prob' in c or 'score' in c: fmt[c] = '{:.3f}'
        elif 'magnitude' in c: fmt[c] = '{:.2f}'
    print(f"Total: {len(candidates)} candidates\n")
    display(candidates[display_cols].head(50).style.format(fmt, na_rep='--'))
else:
    print("No candidates to display")

## Consolidated Broker Report
Each row is a unique event identified by its **Rubin DIA Object ID**, with per-broker classifications shown side by side.

In [ ]:
if candidates is not None and len(candidates) > 0:
    # Build consolidated broker report keyed by Rubin DIA Object ID
    report_rows = []
    for idx, row in candidates.iterrows():
        rubin_id = row.get('rubin_dia_object_id', '')
        if not rubin_id or (isinstance(rubin_id, float) and pd.isna(rubin_id)):
            rubin_id = f"no_rubin_id_{idx}"

        entry = {
            'Rubin DIA ID': rubin_id,
            'ANTARES ID': row.get('object_id_ANTARES', '--'),
            'ALeRCE ID': row.get('object_id_ALeRCE', '--'),
            'RA': row.get('ra'),
            'Dec': row.get('dec'),
            'Discovery': row.get('discovery_date', '--'),
            'Brokers': row.get('brokers_detected', '--'),
        }

        # ANTARES classification
        ant_ia = row.get('classification_ANTARES_ia_prob')
        entry['ANTARES P(Ia)'] = f"{ant_ia:.3f}" if pd.notna(ant_ia) else '--'

        # ALeRCE classification breakdown
        for sn_type, label in [('ia', 'Ia'), ('ii', 'II'), ('ibc', 'Ib/c'), ('slsn', 'SLSN')]:
            col = f'classification_ALeRCE_{sn_type}_prob'
            val = row.get(col)
            entry[f'ALeRCE P({label})'] = f"{val:.3f}" if pd.notna(val) else '--'

        # Combined
        mean_p = row.get('mean_ia_prob')
        entry['Mean P(Ia)'] = f"{mean_p:.3f}" if pd.notna(mean_p) else '--'
        agree = row.get('agreement_score')
        entry['Agreement'] = f"{agree:.2f}" if pd.notna(agree) else '--'

        # Magnitudes
        for broker in ['ANTARES', 'ALeRCE']:
            mag = row.get(f'magnitude_{broker}')
            entry[f'Mag ({broker[:3]})'] = f"{mag:.2f}" if pd.notna(mag) else '--'

        report_rows.append(entry)

    report_df = pd.DataFrame(report_rows)

    # Replace NaN-like strings
    report_df = report_df.fillna('--')
    for col in report_df.columns:
        report_df[col] = report_df[col].apply(
            lambda x: '--' if (isinstance(x, float) and pd.isna(x)) else x)

    print(f"Consolidated Broker Report: {len(report_df)} events\n")
    display(report_df.head(50).style.set_properties(**{
        'text-align': 'center', 'font-size': '11px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '11px')]},
    ]))
else:
    print("No candidates for consolidated report")

## Inspect a CandidateChange `SELECTED_INDEX` and re-run the two cells below to view a different object.

In [ ]:
SELECTED_INDEX = 0
if candidates is not None and len(candidates) > 0:
    c = candidates.iloc[SELECTED_INDEX]
    oid = c.get('object_id_ANTARES') or c.get('object_id_ALeRCE') or '?'
    rubin_id = c.get('rubin_dia_object_id', '')
    prob = c.get('mean_ia_prob')
    print(f"Candidate {SELECTED_INDEX}/{len(candidates)-1}: {oid}")
    if rubin_id:
        print(f"  Rubin DIA Object ID: {rubin_id}")
    print(f"  RA, Dec:  {c['ra']:.5f}, {c['dec']:.5f}")
    print(f"  P(Ia):    {prob:.3f}" if pd.notna(prob) else "  P(Ia):    N/A")
    print(f"  Brokers:  {c.get('brokers_detected', 'N/A')}")
    # Classification comparison
    broker_names = c.get('broker_names', [])
    if isinstance(broker_names, str):
        broker_names = [b.strip() for b in broker_names.split(',')]
    rows = []
    for broker in broker_names:
        row = {'Broker': broker}
        for sn_type in ['ia', 'ii', 'ibc', 'slsn']:
            col = f'classification_{broker}_{sn_type}_prob'
            row[f'P({sn_type.upper()})'] = f"{float(c[col]):.3f}" if col in c.index and pd.notna(c[col]) else '--'
        rows.append(row)
    if rows:
        print()
        display(pd.DataFrame(rows))
    tags = c.get('tags_ANTARES')
    if tags:
        print(f"  ANTARES tags: {tags}")
else:
    print("No candidates available")

In [ ]:
if candidates is not None and len(candidates) > 0:    c = candidates.iloc[SELECTED_INDEX]    # Resolve object ID and broker    broker = 'ANTARES'    oid_col = f'object_id_{broker}'    if oid_col not in c.index or pd.isna(c.get(oid_col)):        broker = 'ALeRCE'        oid_col = f'object_id_{broker}'    if oid_col not in c.index or pd.isna(c.get(oid_col)):        print("No object ID available"); raise SystemExit    object_id = c[oid_col]    print(f"Fetching light curve for {object_id} from {broker}...")    lc = monitor.get_light_curve(object_id, broker=broker)    if lc is not None and len(lc) > 0:        print(f"{len(lc)} photometric points")        if 'survey' in lc.columns:            for s, n in lc['survey'].value_counts().items():                print(f"  {s}: {n} pts")        lc_prep = PlottingUtils.prepare_light_curve(lc)        if lc_prep is not None and len(lc_prep) > 0:            fig = PlottingUtils.plot_light_curve_matplotlib(                lc_prep, title=f"{object_id} ({broker})")            plt.show()            try:                pfig = PlottingUtils.plot_light_curve_plotly(                    lc_prep, title=f"{object_id} ({broker}) [Interactive]")                if pfig: pfig.show()            except: pass        print(f"\nPhotometry (first 20 rows):")        display(lc.head(20))    else:        print(f"No light curve data from {broker}")    # Postage stamps    print(f"\nFetching stamps for {object_id}...")    try:        stamps = monitor.get_stamps(object_id, c['ra'], c['dec'], broker=broker)        if stamps:            print(f"Stamp keys: {list(stamps.keys())}")        else:            print("No stamps available")    except Exception as e:        print(f"Stamps error: {e}")else:    print("No candidates available")

In [ ]:
if candidates is not None and len(candidates) > 0:    outfile = f'type_ia_candidates_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'    export_df = candidates.copy()    for col in export_df.columns:        if export_df[col].apply(lambda x: isinstance(x, list)).any():            export_df[col] = export_df[col].apply(                lambda x: ','.join(str(i) for i in x) if isinstance(x, list) else x)    export_df.to_csv(outfile, index=False)    print(f"Exported {len(candidates)} candidates to {outfile}")else:    print("No candidates to export")

## Reference| Parameter | Default | Description ||---|---|---|| `MIN_IA_PROBABILITY` | 0.5 | Min Type Ia probability || `DAYS_BACK` | 30 | Query window (days) || `MAX_ALERTS` | 200 | Max alerts per broker || `REQUIRE_RUBIN` | True | Only keep loci with Rubin data || `FILTER_ELLIPTICAL` | False | Elliptical host galaxy filter || `MIN_BROKERS` | 1 | Min detecting brokers || `MIN_AGREEMENT` | 0.0 | Min cross-broker agreement |**Surveys**: `ant_survey=1` ZTF detection, `=2` ZTF upper limit (excluded), `=4` Rubin/LSST  **Plots**: colored by band (g/r/i/z), shaped by survey (circle=ZTF, square=Rubin)